# Projeto 1 - T320 (2S2026)

### Instruções:

1. Quando você terminar os exercícios do projeto, vá até o menu do Jupyter ou Colab e selecione a opção para fazer download do notebook.
    * Os notebooks tem extensão .ipynb.
    * Este deve ser o arquivo que você irá entregar.
    * No Colab vá até a opção **File** -> **Download .ipynb**.
2. Após o download do notebook, vá até a aba de tarefas do MS Teams, localize a tarefa referente a este projeto e faça o upload do seu notebook. Veja que há uma opção de anexar arquivos à tarefa.
3. Atente-se ao prazo de entrega definido na tarefa do MS Teams. Entregas fora do prazo não serão aceitas.
4. **O projeto pode ser resolvido em grupos de no MÁXIMO 3 alunos**.
5. Todas as questões têm o mesmo peso.
6. Não se esqueça de colocar seu(s) nome(s) e número(s) de matrícula no campo abaixo.
7. Você pode consultar todo o material de aula.
8. A interpretação faz parte do projeto. Leia o enunciado de cada questão atentamente!
9. Boa sorte!

---



**Nomes e matrículas**:

1.

2.

3.

# 1) **PROJETO SOBRE CLASSIFICAÇÃO MULTICLASSES APLICADA À CARACTERIZAÇÃO DE CANAIS SEM FIO**

**CONTEXTO:**

Você foi contratado por uma empresa de telecomunicações para desenvolver um sistema capaz de identificar automaticamente a condição de propagação de um enlace sem fio.

O sistema recebe diferentes parâmetros medidos pelo receptor e deve classificar o canal em uma das quatro condições abaixo:

* **LOS**: canal com linha de visada;
* **NLOS Leve**: canal sem linha de visada, porém com degradação moderada;
* **NLOS Moderado**: canal com degradação significativa;
* **NLOS Severo**: canal com grande atenuação e forte dispersão.

Para cada enlace serão considerados os seguintes atributos:

| Atributo | Descrição |
|---|---|
| RSRP | Potência média do sinal recebido, em dBm |
| RSRQ | Qualidade do sinal recebido, em dB |
| SINR | Relação sinal-interferência-ruído, em dB |
| Delay Spread | Espalhamento temporal do canal, em ns |
| Doppler | Desvio Doppler, em Hz |
| K-Factor | Fator K do canal Rice, em dB |

Neste exercício serão comparadas duas estratégias para classificação multiclasses:

* Abordagem de classificação **One-versus-Rest (OvR)** com regressores logísticos
* Regressão **Softmax**.

Posteriormente, algumas classes com características semelhantes serão agrupadas e o desempenho será novamente analisado.

 1. Execute a célula abaixo para importar as bibliotecas, gerar artificialmente as amostras e dividir os dados em conjuntos de treinamento e validação.

**DICAS:**

+ Os dados são gerados diretamente pela célula abaixo.
+ São utilizadas 400 amostras de cada condição de propagação.
+ Os dados são separados em 80% para treinamento e 20% para validação. Utiliza-se estratificação para manter a mesma proporção das classes tanto no treino quanto na validação.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, precision_score, recall_score, f1_score

seed = 42
np.random.seed(seed)

# Número de exemplos de cada classe.
N_classe = 400

# Nomes dos atributos.
feature_names = [
    "RSRP",
    "RSRQ",
    "SINR",
    "DelaySpread",
    "Doppler",
    "KFactor"
]

# Média e desvio padrão dos atributos de cada classe.
parametros = {
    0: {
        "media": [-75, -8, 20, 30, 40, 10],
        "std":   [6, 2, 4, 12, 15, 3]
    },

    1: {
        "media": [-85, -11, 13, 80, 60, 5],
        "std":   [7, 2.2, 4, 25, 20, 2.5]
    },

    2: {
        "media": [-95, -14, 6, 150, 85, 1],
        "std":   [8, 2.5, 5, 40, 30, 2.5]
    },

    3: {
        "media": [-104, -17, 0, 230, 110, -2],
        "std":   [8, 2.5, 5, 55, 35, 2.5]
    }
}

X_lista = []
y_lista = []

for classe in range(4):

    media = parametros[classe]["media"]
    std = parametros[classe]["std"]

    Xi = np.column_stack([
        np.random.normal(media[j], std[j], N_classe)
        for j in range(len(feature_names))
    ])

    yi = np.full(N_classe, classe)

    X_lista.append(Xi)
    y_lista.append(yi)

X = np.vstack(X_lista)
y = np.concatenate(y_lista)

# Alguns atributos físicos não podem assumir valores negativos.
X[:, 3] = np.clip(X[:, 3], 1, None)
X[:, 4] = np.clip(X[:, 4], 0, None)

df = pd.DataFrame(X, columns=feature_names)
df["Classe"] = y

# Embaralhamento e divisão.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=seed,
    stratify=y
)

df.head(20)

2. Execute a célula abaixo para verificar as dimensões da base de dados e a distribuição das quatro classes.

In [ ]:
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

print("\nDistribuição das classes:")
print(pd.Series(y).value_counts().sort_index())

nomes_classes = [
    "LOS",
    "NLOS Leve",
    "NLOS Moderado",
    "NLOS Severo"
]

plt.figure(figsize=(8, 5))

bars = plt.bar(
    nomes_classes,
    [np.sum(y == i) for i in range(4)]
)

plt.bar_label(bars)

plt.xlabel("Classe")
plt.ylabel("Número de exemplos")
plt.title("Distribuição das classes")
plt.grid(axis="y")

plt.show()

3. Após analisar a figura acima, responda: a base pode ser considerada balanceada? **(Justifique a resposta)**

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

4. Padronize os atributos dos conjuntos de treinamento e validação.

**DICA**
+ Não se esqueça que os parâmetros para a padronização devem ser calculados com o conjunto de treinamento e usados para padronizar os conjuntos de treinamento e validação.

In [ ]:
# Digite aqui o código do exercício.

5. Classifique os dados utilizando a estratégia **One-versus-Rest (OvR)** com regressores logísticos. Utilize os conjuntos de treinamento e validação padronizados.

Após o treinamento:

1. realize as predições sobre o conjunto de validação;
2. calcule a acurácia.

**DICAS:**

+ Utilize o código abaixo para instanciar o classificar usando a estratégia **One-versus-Rest (OvR)**:
```python
modelOvR = OneVsRestClassifier(
    LogisticRegression(
        random_state=seed
    )
)
```
+ O treinamento e as predições são feitos com os métodos `fit` e `predict`, respectivamente.

In [ ]:
# Digite aqui o código do exercício.

6. Quantos regressores logísticos são treinados quando se usa a abordagem **One-versus-Rest (OvR)**? **(Justifique sua resposta)**

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

7. Calcule e plote a matriz de confusão do classificador OvR utilizando o conjunto de validação.

Utilize os nomes abaixo para as linhas e colunas da matriz:
* LOS;
* NLOS Leve;
* NLOS Moderado;
* NLOS Severo.

In [ ]:
# Digite aqui o código do exercício.

 8. Analise a matriz de confusão obtida. Quais condições de propagação foram confundidas com maior frequência? Existe uma relação física entre as classes que foram confundidas? **(Justifique as respostas)**

**DICAS:**

+ Observe principalmente classes que representam condições de canal semelhantes.
+ Analise a diferença entre NLOS Leve, Moderado e Severo.

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

9. Utilize a função `classification_report` para apresentar:

* precisão;
* recall;
* F1-score;
* quantidade de exemplos.

Apresente as métricas das quatro classes.

In [ ]:
# Digite aqui o código do exercício.

10. Analise o reporte de classificação acima e responda:

+ Qual classe apresenta o pior desempenho em termos de F1-score?
+ Por qual motivo ela tem esse desempenho ruim?

**(Justifique suas respostas)**

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

11. Classifique agora os mesmos dados utilizando **Regressão Softmax**. Utilize os conjuntos de treinamento e validação padronizados.

Após o treinamento:

1. realize as predições sobre o conjunto de validação;
2. calcule a acurácia.

**DICAS:**

+ Utilize o código abaixo para instanciar o classificar usando o regressor *softmax*:
```python
modelSM LogisticRegression(
    random_state=seed
)
```
+ A classe `LogisticRegression` implementa o regressor *softmax* quando o número de classes é maior do que 2 (i.e., $Q>2$).
+ O treinamento e as predições são feitos com os métodos `fit` e `predict`, respectivamente.

In [ ]:
# Digite aqui o código do exercício.

12. Calcule e plote a matriz de confusão obtida pelo regressor Softmax.

In [ ]:
# Digite aqui o código do exercício.

 13. Utilize `classification_report` para apresentar as métricas do classificador Softmax.

In [ ]:
# Digite aqui o código do exercício.

14. Compare os classificadores **OvR e Softmax**. Qual apresentou o melhor desempenho? **(Justifique sua resposta)**

Analise:

* acurácia;
* precisão;
* recall;
* F1-score;
* matriz de confusão.

Não analise apenas o desempenho global. Observe também o comportamento para cada classe individualmente.

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

15. As classes **NLOS Moderado** e **NLOS Severo** representam situações em que não existe linha de visada e o enlace apresenta uma degradação significativa.

Considere que, para determinada aplicação, não seja necessário diferenciar esses dois casos.

Vamos agrupar:

* NLOS Moderado;
* NLOS Severo;

em uma única classe chamada **NLOS Crítico**.

Após o agrupamento teremos:

* Classe 0 → LOS;
* Classe 1 → NLOS Leve;
* Classe 2 → NLOS Crítico.

Execute a célula abaixo para agrupar as duas classes.

In [ ]:
# ============================================================
# Agrupamento balanceado das classes 2 e 3
# ============================================================

# Número de amostras que cada classe possui após o agrupamento.
# Classe 0: 320
# Classe 1: 320
# Portanto, a nova classe NLOS Crítico também deve ter 320.

rng = np.random.default_rng(seed)

# ------------------------------------------------------------
# TREINAMENTO
# ------------------------------------------------------------

# Índices das amostras das classes 2 e 3
idx_classe2_train = np.where(y_train == 2)[0]
idx_classe3_train = np.where(y_train == 3)[0]

# Número de amostras de cada classe que serão utilizadas
# para formar a nova classe.
n_amostras_train = len(idx_classe2_train) // 2

# Seleciona aleatoriamente metade da classe 2
idx_2_selecionados_train = rng.choice(
    idx_classe2_train,
    size=n_amostras_train,
    replace=False
)

# Seleciona aleatoriamente metade da classe 3
idx_3_selecionados_train = rng.choice(
    idx_classe3_train,
    size=n_amostras_train,
    replace=False
)

# Índices que permanecerão no conjunto de treinamento
idx_train_final = np.concatenate([
    np.where((y_train == 0) | (y_train == 1))[0],
    idx_2_selecionados_train,
    idx_3_selecionados_train
])

# Cria o novo conjunto de treinamento
X_train = X_train[idx_train_final]
y_train = y_train[idx_train_final]

# Todas as amostras selecionadas das classes 2 e 3
# passam a pertencer à classe 2 (NLOS Crítico).
y_train[(y_train == 3)] = 2

# ------------------------------------------------------------
# VALIDAÇÃO
# ------------------------------------------------------------

# Índices das amostras das classes 2 e 3
idx_classe2_test = np.where(y_test == 2)[0]
idx_classe3_test = np.where(y_test == 3)[0]

# Metade das amostras de cada classe
n_amostras_test = len(idx_classe2_test) // 2

# Seleciona aleatoriamente metade da classe 2
idx_2_selecionados_test = rng.choice(
    idx_classe2_test,
    size=n_amostras_test,
    replace=False
)

# Seleciona aleatoriamente metade da classe 3
idx_3_selecionados_test = rng.choice(
    idx_classe3_test,
    size=n_amostras_test,
    replace=False
)

# Índices que permanecerão no conjunto de validação
idx_test_final = np.concatenate([
    np.where((y_test == 0) | (y_test == 1))[0],
    idx_2_selecionados_test,
    idx_3_selecionados_test
])

# Cria o novo conjunto de validação
X_test = X_test[idx_test_final]
y_test = y_test[idx_test_final]

# Classe 3 selecionada também passa a ser classe 2
y_test[y_test == 3] = 2

# ------------------------------------------------------------
# EMBARALHAR OS CONJUNTOS
# ------------------------------------------------------------

# Embaralha treinamento
ordem_train = rng.permutation(len(y_train))
X_train = X_train[ordem_train]
y_train = y_train[ordem_train]

# Embaralha validação
ordem_test = rng.permutation(len(y_test))
X_test = X_test[ordem_test]
y_test = y_test[ordem_test]

# ------------------------------------------------------------
# VERIFICAÇÃO
# ------------------------------------------------------------

print("Classes de treinamento:", np.unique(y_train))
print("Classes de validação:", np.unique(y_test))

print("\nDistribuição - treinamento:")

unique, counts = np.unique(y_train, return_counts=True)

for classe, quantidade in zip(unique, counts):
    print(f"Classe {classe}: {quantidade} exemplos")

print("\nDistribuição - validação:")

unique, counts = np.unique(y_test, return_counts=True)

for classe, quantidade in zip(unique, counts):
    print(f"Classe {classe}: {quantidade} exemplos")

16. Utilizando o classificador que apresentou o melhor desempenho anteriormente, refaça o treinamento utilizando agora as três classes. Calcule a nova acurácia.

**DICA**
+ Instancie outro objeto da classe do melhor classificador.

In [ ]:
# Digite aqui o código do exercício.

17. Plote a nova matriz de confusão.

Utilize os nomes abaixo para as linhas e colunas da matriz:

* LOS;
* NLOS Leve;
* NLOS Crítico.

In [ ]:
# Digite aqui o código do exercício.

18. Utilize `classification_report` para apresentar as métricas com o melhor modelo e o novo número de classes.

In [ ]:
# Digite aqui o código do exercício.

 19. Compare os resultados obtidos quando o problema possuía quatro classes com os resultados obtidos após o agrupamento em três classes.

Analise:

* acurácia;
* precisão;
* recall;
* F1-score;
* matriz de confusão.

O agrupamento facilitou o problema de classificação? (**Justifique sua resposta.**)


**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

# 2) **PROJETO SOBRE CLASSIFICAÇÃO APLICADA À DETECÇÃO DE SÍMBOLOS DE UMA MODULAÇÃO DIGITAL**

**CONTEXTO:**

Você é um engenheiro de telecomunicações e foi contratado para desenvolver um sistema de demodulação inteligente utilizando Machine Learning.

Neste projeto será considerada uma modulação digital **16-QAM**.

Na modulação 16-QAM existem 16 símbolos possíveis na constelação.

Cada símbolo será considerado como uma classe diferente.

O sinal transmitido será afetado por um canal AWGN (*Additive White Gaussian Noise*).

O classificador deverá receber como atributos:

* componente em fase \(I\);
* componente em quadratura \(Q\);

e determinar qual dos 16 símbolos foi transmitido.

1. Execute a célula abaixo para:

+ gerar símbolos 16-QAM;
+ adicionar ruído AWGN aos símbolos;
+ gerar a matriz de atributos;
+ dividir os dados em conjuntos de treinamento e validação;
+ visualizar a constelação recebida.

In [ ]:
# Import all necessary libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, auc, classification_report
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from scipy.special import erfc

# Reset PN sequence random generator.
seed = 42
np.random.seed(seed)

# Mapping table.
mapping_table = [-3-3j, -3-1j, -3+3j, -3+1j, -1-3j, -1-1j, -1+3j, -1+1j, 3-3j, 3-1j, 3+3j, 3+1j, 1-3j, 1-1j, 1+3j, 1+1j]

# Modulate bits into 16QAM symbols.
def mod(bits):
    symbols = np.zeros((len(bits), 1), dtype=complex)
    for i in range(0, len(bits)):
        symbols[i, 0] = mapping_table[bits[i]]/np.sqrt(10.0)
    return symbols

# Number of symbols to be transmitted.
N = 10000

# Number of classes.
numOfClasses = 16

# Create Es/N0 vector.
EsN0dB = 20
EsN0Lin = 10.0**(-(EsN0dB/10.0))

# Generate bits.
bits_16qam = np.random.randint(0, numOfClasses, N)
# Modulate the binary stream into 16QAM symbols.
symbols = mod(bits_16qam)

# Generate noise vector.
# Divide by two since the theoretical ber uses a complex Normal pdf with variance of each part = 1/2.
noise = np.sqrt(EsN0Lin/2.0)*(np.random.randn(N, 1) + 1j*np.random.randn(N, 1))

# Pass 16QAM symbols through AWGN channel.
y = symbols + noise

# Create the attribute matrix.
X = np.c_[np.real(y), np.imag(y)]

# Split array into random train and test subsets.
X_train, X_test, y_train, y_test = train_test_split(X, bits_16qam, test_size=0.3, random_state=seed)

# Plot the classes.
for i in range(numOfClasses):
    idx = np.argwhere(bits_16qam == i)
    label = 'Symbol '+str(i)
    plt.plot(np.real(y[idx.ravel()]), np.imag(y[idx.ravel()]), '.', label=label)
plt.grid()
plt.xlabel('InPhase')
plt.ylabel('Quadrature')
plt.legend(bbox_to_anchor=(1.1, 1.05))
plt.show()

2. Padronize os atributos dos conjuntos de treinamento e validação.

**DICA**
+ Não se esqueça que os parâmetros para a padronização devem ser calculados com o conjunto de treinamento e usados para padronizar os conjuntos de treinamento e validação.

Utilize:

```python
StandardScaler()
```

In [ ]:
# Digite aqui o código do exercício.

3. Treine um classificador Softmax para identificar os 16 símbolos da modulação 16-QAM.

Utilize:
```python
model = LogisticRegression(
    random_state=seed
)
```

In [ ]:
# Digite aqui o código do exercício.

4. Calcule e plote a matriz de confusão do classificador para o conjunto de validação.

In [ ]:
# Digite aqui o código do exercício.

5. Utilize a função `classification_report` para apresentar as métricas das 16 classes. Use o conjunto de validação.

**DICA**
+ Use:
```python
cr = classification_report(y_test, y_pred, digits=4)
```

In [ ]:
# Digite aqui o código do exercício.

6. Plote as regiões de decisão do classificador. Sobre as regiões de decisão, plote também todas as amostras geradas neste exercício.

**DICA**:
+ Concatene as matrizes de atributos padronizados e os vetores de rótulos de treinamento e validação.



In [ ]:
# Digite aqui o código do exercício.

 7. Analise as regiões de decisão. As fronteiras aprendidas pelo classificador são coerentes com a geometria da constelação 16-QAM? Onde ocorre a maior parte dos erros de classificação? Explique por que símbolos vizinhos possuem maior probabilidade de serem confundidos. **(Justifique sua resposta)**

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

8. Execute a célula abaixo para comparar a Taxa de Erro de Símbolo (SER) obtida pelo classificador com a SER teórica de uma modulação 16-QAM em canal AWGN.

**IMPORTANTE:**

Para esta célula funcionar, o classificador treinado anteriormente deve possuir o nome:

```python
model

In [ ]:
# Number of symbols to be transmitted.
N = 10000000

# Create Es/N0 vector.
EsN0dB = np.arange(0, 22, 2)

# Iterate over all EsN0 values and calculate SER.
ser_simu = np.zeros((len(EsN0dB),))
ser_theo = np.zeros((len(EsN0dB),))
for idx in range(len(EsN0dB)):
    EsN0Lin = 10.0**(-(EsN0dB[idx]/10.0))

    # Generate 16QAM symbols.
    bits_16qam = np.random.randint(0, numOfClasses, N)
    # Modulate the binary stream into 16QAM symbols.
    symbol = mod(bits_16qam)

    # Pass QPSK symbols through AWGN channel.
    noise = np.sqrt(EsN0Lin/2.0)*(np.random.randn(N, 1) + 1j*np.random.randn(N, 1))
    y = symbol + noise

    # Detect received symbol.
    X = np.c_[np.real(y), np.imag(y)]
    X = scaler.transform(X)
    detected_symbol = model.predict(X)

     # Simulated 16QAM SER.
    ser_simu[idx] = sum(detected_symbol != bits_16qam)/N

    # Theoretical 16QAM BER.
    M = 16
    k = np.sqrt(3/(2*(M-1)))
    EsN0 = 10.0**(EsN0dB[idx]/10.0)
    ser_theo[idx] = 2*(1 - (1/np.sqrt(M)))*erfc(k*np.sqrt(EsN0)) - (1 - (2/np.sqrt(M)) + (1/M))*(erfc(k*np.sqrt(EsN0)))**2.0

    # Print Es/N0 versus BER values.
    print('Es/N0:%d \t- SER simu: %e \t- SER theo: %e' % (EsN0dB[idx], ser_simu[idx], ser_theo[idx]))

plt.plot(EsN0dB, ser_theo, label='theoretical')
plt.plot(EsN0dB, ser_simu, 'ro', label='simulated')
plt.xlabel('Es/N0 [dB]')
plt.ylabel('SER')
plt.xscale('linear')
plt.yscale('log')
plt.grid()
plt.title('16QAM detection')
plt.legend()
plt.xlim([0, 20.5])
plt.ylim([1e-5, 1])
plt.show()

9. Analise o gráfico anterior e responda:

+ O que acontece com a SER quando a relação $(E_s/N_0)$ aumenta?
+ O desempenho do classificador se aproxima do detector teórico?
+ Por que a taxa de erro aumenta para valores baixos de $(E_s/N_0)$?
+ Relacione esse comportamento com a constelação apresentada anteriormente.

**(Justifique suas respostas)**

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

# 3) **PROJETO DE CLASSIFICAÇÃO APLICADA À DETECÇÃO DE FALHAS EM ENLACES DE TELECOMUNICAÇÕES**

**CONTEXTO:**

Uma operadora de telecomunicações deseja prever situações de falha ou indisponibilidade (*outage*) em enlaces móveis.

O objetivo é desenvolver um classificador capaz de determinar se um enlace apresenta funcionamento normal ou se está próximo de uma condição de falha.

Serão utilizados os seguintes atributos:

| Atributo | Descrição |
|---|---|
| RSRP | Potência do sinal recebido |
| RSRQ | Qualidade do sinal recebido |
| SINR | Relação sinal-interferência-ruído |
| Latência | Tempo de resposta da rede, em ms |
| Jitter | Variação do atraso, em ms |
| Packet Loss | Percentual de pacotes perdidos |
| Usuários | Número de usuários conectados |

A variável alvo será:

* **0 → Enlace normal**
* **1 → Outage/Falha**

Em redes reais, falhas costumam ocorrer com uma frequência muito menor que situações normais de operação.

Por esse motivo, este exercício também irá explorar os problemas relacionados a **bases de dados desbalanceadas**.

**OBJETIVO**:
Desenvolver e avaliar classificadores para detecção de uma classe rara, analisando como o desbalanceamento influencia as métricas e como diferentes estratégias podem melhorar a detecção da classe minoritária.

 1. Execute a célula abaixo para gerar artificialmente os dados do enlace.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import seaborn as sns

seed = 42
np.random.seed(seed)

N_normal = 45000
N_outage = 500

def gerar_dados_enlace(
    N,
    medias,
    desvios
):

    return np.column_stack([
        np.random.normal(
            medias[i],
            desvios[i],
            N
        )
        for i in range(len(medias))
    ])

# Operação normal.
X_normal = gerar_dados_enlace(
    N_normal,

    medias=[
        -88,
        -11,
        14,
        40,
        10,
        1.5,
        40
    ],

    desvios=[
        8,
        2.5,
        5,
        15,
        5,
        1.2,
        18
    ]
)

# Situação de outage.
X_outage = gerar_dados_enlace(
    N_outage,

    medias=[
        -93,    # RSRP
        -13,    # RSRQ
        10,     # SINR
        55,     # Latencia
        15,     # Jitter
        3.0,    # PacketLoss
        50      # Usuarios
    ],

    desvios=[
        10,
        3.5,
        7,
        25,
        10,
        2.5,
        22
    ]
)

X3 = np.vstack([
    X_normal,
    X_outage
])

y3 = np.concatenate([
    np.zeros(N_normal),
    np.ones(N_outage)
]).astype(int)

# Valores que fisicamente não podem ser negativos.
X3[:, 3] = np.clip(
    X3[:, 3],
    0,
    None
)

X3[:, 4] = np.clip(
    X3[:, 4],
    0,
    None
)

X3[:, 5] = np.clip(
    X3[:, 5],
    0,
    100
)

X3[:, 6] = np.clip(
    X3[:, 6],
    0,
    None
)

features3 = [
    "RSRP",
    "RSRQ",
    "SINR",
    "Latencia",
    "Jitter",
    "PacketLoss",
    "Usuarios"
]

df3 = pd.DataFrame(
    X3,
    columns=features3
)

df3["Outage"] = y3

df3.head(20)

2. Execute a célula abaixo para visualizar a distribuição das classes.

In [ ]:
plt.figure(figsize=(7, 5))

bars = plt.bar(
    ["Normal", "Outage"],
    [
        np.sum(y3 == 0),
        np.sum(y3 == 1)
    ]
)

plt.bar_label(bars)

plt.xlabel("Classe")
plt.ylabel("Número de exemplos")
plt.title("Distribuição das classes")

plt.grid(axis="y")

plt.show()

 3. Após analisar a figura anterior, responda:

+ O conjunto de dados é balanceado ou desbalanceado?
+ A acurácia, isoladamente, seria uma boa métrica para avaliar o classificador?
+ Qual seria o problema de um classificador que classificasse praticamente todas as amostras como "Normal"?
+ Neste problema, qual métrica você considera especialmente importante para avaliar o desempenho do classificador com relação à classe Outage (ou seja, a classficação incorreta de amostras da classe Outage tem um alto custo para o sistema e deve ser minimizada)?

(**Justifique suas respostas.**)

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

4. Divida os dados em três conjuntos:

* 60% para treinamento;
* 20% para validação;
* 20% para teste.

**Use estratificação durante a criação dos três conjuntos. Para isso, passe o parâmetro `stratify` com o vetor de rótulos para a função `train_test_split`.**

Utilize:

```python
random_state=seed
```

**DICA**
+ Os nomes das variáveis contendo os atributos e rótulos são: `X3` e `y3`.

In [ ]:
# Digite aqui o código do exercício.

5. Padronize os atributos dos conjuntos de treinamento, validação e teste.

**DICA**
+ Não se esqueça que os parâmetros para a padronização devem ser calculados com o conjunto de treinamento e usados para padronizar os conjuntos de treinamento, validação e teste.

Utilize:

```python
StandardScaler()
```

In [ ]:
# Digite aqui o código do exercício.

6. Vamos inicialmente construir um **modelo baseline**, sem nenhuma técnica específica para lidar com o desbalanceamento. Treine inicialmente um Regressor Logístico.

Utilize:
```python
LogisticRegression(
    random_state=seed
)
```

In [ ]:
# Digite aqui o código do exercício.

7. Plote a matriz de confusão do classificador anterior. Use o conjunto de teste.

Considere:
* Classe 0 → Normal;
* Classe 1 → Outage.

In [ ]:
# Digite aqui o código do exercício.

8. Utilize a função `classification_report` para apresentar as métricas das 2 classes. Use o conjunto de teste.

In [ ]:
# Digite aqui o código do exercício.

9. Analise os resultados anteriores. É possível possuir uma acurácia elevada e, ao mesmo tempo, um desempenho ruim na detecção de outages? Utilize principalmente os valores de:

* recall da classe Outage;
* precisão;
* F1-score;
* falsos negativos.

Explique por que um **falso negativo** pode ser especialmente problemático para uma operadora.

**(Justifique suas respostas)**

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

10. Treine novamente o Regressor Logístico, mas desta vez utilizando o parâmetro:

```python
class_weight="balanced"
```

Utilize:
```python
model = LogisticRegression(
    class_weight="balanced",
    random_state=seed
)
```

O objetivo do classificador treinado neste exercício é o de identificar *outage* (classe positva), mas nós não temos muitos desses exemplos desta classe. Então, nós gostariamos que o classificador **ponderasse fortemente** (ou seja, atribuísse um peso maior) os poucos exemplos positivos disponíveis.

Isso pode ser feito configurando-se o parâmetro `class_weight` da classe `LogisticRegression` com a string `'balanced'`. Essa configuração fará com que o modelo **preste mais atenção** aos exemplos de uma classe sub-representada, ou seja, com poucos exemplos.

O modo `balanced` usa os valores de `y` para calcular automaticamente os pesos de cada classe. Os pesos são inversamente proporcionais às frequências de cada classe nos valores de `y`. O peso de cada classe é calculado através da seguinte equação: `n_samples / (n_classes * np.bincount(y))`, onde `n_samples` é o número total de exemplos, `n_classes` é o número de classes e `np.bincount(y)` retorna o número de exemplos de cada uma das classes.

Portanto, de posse destas informações, treine um novo regressor logísitco que use os pesos das classes para melhorar seu desempenho, atribuindo maior peso aos exemplos da classe minoritária.

In [ ]:
# Digite aqui o código do exercício.

11. Plote a matriz de confusão do modelo treinado utilizando `class_weight="balanced"`. Use o conjunto de teste.

In [ ]:
# Digite aqui o código do exercício.

12. Utilize a função `classification_report` para apresentar as métricas das 2 classes. Use o conjunto de teste.

In [ ]:
# Digite aqui o código do exercício.

13. Compare o modelo baseline com o modelo treinado utilizando `class_weight="balanced"`.

Analise:

* acurácia;
* precisão;
* recall;
* F1-score;
* número de falsos positivos;
* número de falsos negativos.

Responda:

+ Qual dos dois modelos seria o mais adequado para detectar situações de outage?
+ O modelo utilizando `class_weight="balanced"` reduziu o número de falsos negativos?
+ O que aconteceu com o número de falsos positivos quando comparado ao modelo baseline?
+ Por que aumentar a atenção dada à classe Outage pode aumentar também o número de falsos alarmes (i.e., falsos positivos)?

**(Justifique suas respostas)**


**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

14. Por padrão, em uma classificação binária, o regressor logístico utiliza um limiar de $0.5$: uma amostra é classificada como Outage quando a probabilidade estimada de pertencer à classe 1 é maior ou igual a $0.5$.

Neste exercício, vamos investigar se esse limiar é o mais adequado para o problema de detecção de outage.

O código abaixo retreina o modelo baseline e testa diferentes limiares entre: $0.01$ e $0.99$ em incrementos de $0.001$ e encontra o limiar que maximiza o F1-score da classe Outage.

Portanto, execute o código abaixo e analise os resultados.

O código:

+ Plota um gráfico de F1-score em função do limiar de quantização.
+ Imprime os valores do melhore limiar e F1-score.
+ Imprime o reporte de classificação usando o melhor limiar de quantização encontrado e o conjunto de teste.
+ Plota a matriz de confusão usando o melhor limiar de quantização encontrado e o conjunto de teste (OBS.: classes verdadeiras nas colunas e classes preditas nas  linhas).

**IMPORTANTE:**

+ O código abaixo usa o conjunto de validação para escolher o melhor limiar de decisão.

In [ ]:
# Treinamos o modelo baseline novamente.
baseline = LogisticRegression(
    random_state=seed,
)

baseline.fit(X_train, y_train)

y_pred = baseline.predict_proba(X_val)

thresholds = np.arange(0.01, 0.99, 0.001)

f1_scores = []

for threshold in thresholds:
    y_pred_binary = (y_pred[:, 1] >= threshold).astype(int)
    cm = confusion_matrix(y_val, y_pred_binary)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * (precision * recall) / (precision + recall)
    f1_scores.append(f1)

print('OTIMIZAÇÃO do LIMIAR COM O CONJUNTO DE VALIDAÇÃO')
plt.plot(thresholds, f1_scores)
plt.xlabel("Threshold")
plt.ylabel("F1-score otimizado no conjunto de validação")
plt.title("F1-score vs Threshold")
plt.show()

best_threshold = thresholds[np.argmax(f1_scores)]
print("Best threshold:", best_threshold)
print("Best F1-score:", np.max(f1_scores))
print('\n')

print('RESULTADOS COM O CONJUNTO DE TESTE')
y_pred = baseline.predict_proba(X_test)

y_pred_binary = (y_pred[:, 1] >= best_threshold).astype(int)

cm = confusion_matrix(y_test, y_pred_binary)

sns.heatmap(
    cm.T,
    annot=True,
    fmt="d",
    xticklabels=["Normal", "Outage"],
    yticklabels=["Normal", "Outage"]
)

cr = classification_report(
    y_test,
    y_pred_binary,
    target_names=["Normal", "Outage"]
)

print(cr)

15. Analisando os resultados do item anterior, responda:

+ O limiar padrão de um classificador logístico é $0.5$. No item anterior, o limiar encontrado foi aproximadamente $0.329$. O que significa, intuitivamente, reduzir o limiar de $0.5$ para $0.329$?

+ Por que um limiar de $0.329$ faz com que uma amostra tenha maior probabilidade de ser classificada como *Outage*?

+ Quantos outages existentes no conjunto de teste deixaram de ser detectados pelo modelo?

+ Como o modelo consegue apresentar aproximadamente 99% de acurácia mesmo apresentando desempenho consideravelmente inferior na detecção de Outage?

+ Por que seria inadequado escolher o threshold que maximiza o F1-score diretamente utilizando o conjunto de teste?  

**(Justifique todas as respostas)**

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>

16. Uma possibilidade para lidar com bases desbalanceadas é criar artificialmente exemplos da classe minoritária. Neste exercício será utilizado o algoritmo **Borderline Synthetic Minority Over-sampling Technique (BorderlineSMOTE)**.

**BorderlineSMOTE - tratamento de classes desbalanceadas**

O **BorderlineSMOTE** é uma técnica de *oversampling* utilizada quando uma classe possui muito menos exemplos que outra. Ele procura identificar os exemplos da classe minoritária que estão **próximos da fronteira de decisão entre as classes**, ou seja, regiões em que é mais difícil distinguir uma classe da outra.

Em vez de gerar novas amostras da classe minoritária de forma indiscriminada, o BorderlineSMOTE concentra a geração de exemplos sintéticos nessas regiões de fronteira. Para isso, analisa os vizinhos mais próximos de cada exemplo minoritário e identifica aqueles que estão em uma situação de "perigo", isto é, cercados por uma quantidade significativa de exemplos da classe majoritária. Novas amostras são então geradas a partir desses exemplos e de seus vizinhos da classe minoritária.

Neste exercício, utilizaremos o BorderlineSMOTE para gerar exemplos sintéticos de **Outage**, permitindo que o modelo tenha mais informações sobre as regiões em que **Normal** e **Outage** são mais difíceis de distinguir.

Neste exercício, o objetivo do BorderlineSMOTE não é aumentar artificialmente a quantidade de dados disponíveis para avaliação, mas fornecer ao modelo de treinamento mais exemplos da classe minoritária. O desempenho final deve continuar sendo medido em dados de validação/teste que não foram modificados pelo BorderlineSMOTE.

**Importante:** o BorderlineSMOTE deve ser aplicado **somente aos dados de treinamento**. Os conjuntos de validação e teste devem permanecer com sua distribuição original, pois eles devem representar os dados reais utilizados para avaliar o desempenho do modelo.

Execute a célula abaixo para aplicar o BorderlineSMOTE ao conjunto de treinamento. As distribuições das classes antes e após a aplicação do BorderlineSMOTE são apresentadas.

In [ ]:
import imblearn
from imblearn.over_sampling import BorderlineSMOTE
from collections import Counter

bSMOTE = BorderlineSMOTE(random_state=seed)

X_train_bsmote, y_train_bsmote = bSMOTE.fit_resample(X_train, y_train)

print(f'Dimensões originais do dataset: {Counter(y_train)}')
print(f'Dimensões do dataset reamostrado: {Counter(y_train_bsmote)}')

# Cria uma figura com dois gráficos lado a lado
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1 - Dataset original
bars = ax1.bar(
    ["Normal", "Outage"],
    [
        np.sum(y_train == 0),
        np.sum(y_train == 1)
    ]
)

ax1.bar_label(bars)

ax1.set_xlabel("Classe")
ax1.set_ylabel("Número de exemplos")
ax1.set_title("Distribuição das classes\nno conjunto de treinamento original")

ax1.grid(axis="y")

# Gráfico 2 - Dataset após BorderlineSMOTE
bars = ax2.bar(
    ["Normal", "Outage"],
    [
        np.sum(y_train_bsmote == 0),
        np.sum(y_train_bsmote == 1)
    ]
)

ax2.bar_label(bars)

ax2.set_xlabel("Classe")
ax2.set_ylabel("Número de exemplos")
ax2.set_title("Distribuição das classes\napós a reamostragem com BorderlineSMOTE")

ax2.grid(axis="y")

# Ajustar automaticamente o espaçamento
plt.tight_layout()

plt.show()

17. Treine um novo Regressor Logístico utilizando o conjunto de treinamento após a aplicação do BorderlineSMOTE. Nomeie o novo modelo como `bsmote_model`.

Utilize:
```python
bsmote_model = LogisticRegression(
    random_state=seed
)
```

**Importante**: Os nomes das variáveis do conjunto de treinamento após a aplicação do BorderlineSMOTE são: `X_train_bsmote`  e `y_train_bsmote`.

In [ ]:
# Digite aqui o código do exercício.

18. Plote a matriz de confusão do modelo treinado utilizando BorderlineSMOTE. Use o conjunto de teste.

In [ ]:
# Digite aqui o código do exercício.

19. Utilize a função `classification_report` para apresentar as métricas das 2 classes com o modelo treinado utilizando BorderlineSMOTE. Use o conjunto de teste.

In [ ]:
# Digite aqui o código do exercício.

20. Compare agora as estratégias:

+ Regressão Logística baseline;
+ Regressão Logística com `class_weight="balanced"`;
+ Regressão Logística com limiar otimizado;
+ Regressão Logística utilizando BorderlineSMOTE.

Responda:

* Qual apresentou a maior acurácia?
* Qual apresentou o maior recall para a classe Outage?
* Qual apresentou o maior F1-score para a classe Outage?
* Qual apresentou menor número de falsos negativos?
* Suponha que perder um outage (i.e., falso negativo) custe 100 vezes mais que um falso alarme (i.e., falso positivo). Qual dos quatro modelos minimiza o custo (100·FN + FP)? E se o custo relativo fosse 1 para 1?
* Qual modelo você escolheria para ser utilizado pela operadora? Lembre-se de que uma falha não identificada pode ser mais prejudicial do que um falso alarme. Se cada alarme aciona uma equipe, o excesso de alertas leva à fadiga e ao descrédito do sistema.

(**Justifique suas respostas.**)

**Resposta**

<span style="color:blue">Digite abaixo a resposta do exercício.</span>